# Calibration and decision-curve analysis: worked solution

Worked solution for the exercise in the lesson **[Calibration and decision-curve analysis](https://ramkumarmd.com/medical-ai/machine-learning/calibration-and-decision-curves)** (*Machine learning* track, Medical AI course by Dr. Ram Kumar G). Try the [exercise notebook](08-calibration-and-decision-curves.ipynb) first. Other answers can be just as good: the written parts show one reasonable answer, not the only one.

**In short:** Calibration asks whether predicted risks match what actually happens; decision-curve analysis asks whether using the model would lead to better decisions than treating everyone or no one. When a model moves to a new hospital, it often keeps its ranking ability (AUROC) but its risks become too high or too low. Recalibration with a small local sample can fix this, and net benefit shows whether the model is worth using at the thresholds that matter clinically.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### Why calibration fails at new sites

Develop at one synthetic site; apply to another with lower baseline risk.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

predictors = ["age", "bmi", "fasting_glucose", "family_history", "hypertension"]

def simulate(n, baseline, seed):
    rng = np.random.default_rng(seed)
    d = pd.DataFrame({
        "age": rng.integers(30, 75, n),
        "bmi": rng.normal(27, 4.5, n).round(1),
        "fasting_glucose": rng.normal(100, 10, n).round(),
        "family_history": rng.binomial(1, 0.3, n),
        "hypertension": rng.binomial(1, 0.35, n),
    })
    logit = (baseline + 0.03 * d.age + 0.10 * d.bmi + 0.11 * d.fasting_glucose
             + 0.6 * d.family_history + 0.4 * d.hypertension)
    d["diabetes_5y"] = rng.binomial(1, 1 / (1 + np.exp(-logit)))
    return d

development = simulate(5000, baseline=-17.5, seed=5)
new_site = simulate(5000, baseline=-18.5, seed=6)        # lower baseline risk
model = LogisticRegression(penalty=None, max_iter=1000)
model.fit(development[predictors], development.diabetes_5y)

p_new = model.predict_proba(new_site[predictors])[:, 1]
print(f"New site AUROC {roc_auc_score(new_site.diabetes_5y, p_new):.3f}")
print(f"Mean predicted {p_new.mean():.1%}, observed {new_site.diabetes_5y.mean():.1%}")

groups = pd.qcut(p_new, 5, labels=False)
table = pd.DataFrame({"predicted": p_new, "observed": new_site.diabetes_5y}).groupby(groups).mean()
print((table * 100).round(1).rename_axis("risk fifth"))

# New site AUROC 0.806
# Mean predicted 18.9%, observed 9.4%
#             predicted  observed
# risk fifth
# 0                 2.6       1.0
# 1                 6.8       2.8
# 2                12.8       5.6
# 3                23.0       9.9
# 4                49.3      27.8

### Recalibrating with local data

Continuing from the previous block: recalibrate on 2,000 patients from the new site, evaluate on the other 3,000.

In [ ]:
recal_part, eval_part = new_site.iloc[:2000], new_site.iloc[2000:]

def log_odds(d):
    p = model.predict_proba(d[predictors])[:, 1]
    return np.log(p / (1 - p)).reshape(-1, 1)

recalibrator = LogisticRegression(penalty=None).fit(log_odds(recal_part), recal_part.diabetes_5y)
p_before = model.predict_proba(eval_part[predictors])[:, 1]
p_after = recalibrator.predict_proba(log_odds(eval_part))[:, 1]
print(f"Recalibration: slope {recalibrator.coef_[0][0]:.2f}, "
      f"intercept {recalibrator.intercept_[0]:.2f}")
print(f"Observed {eval_part.diabetes_5y.mean():.1%}; mean predicted before "
      f"{p_before.mean():.1%}, after {p_after.mean():.1%}")
print(f"AUROC before {roc_auc_score(eval_part.diabetes_5y, p_before):.3f}, "
      f"after {roc_auc_score(eval_part.diabetes_5y, p_after):.3f}")

# Recalibration: slope 0.93, intercept -1.01
# Observed 9.1%; mean predicted before 18.7%, after 9.6%
# AUROC before 0.812, after 0.812

### Net benefit: is the model worth using?

Continuing from the previous block: net benefit of the model before and after recalibration.

In [ ]:
y = eval_part.diabetes_5y.to_numpy()

def net_benefit(p, threshold):
    flagged = p >= threshold
    tp = np.sum(flagged & (y == 1)) / len(y)
    fp = np.sum(flagged & (y == 0)) / len(y)
    return tp - fp * threshold / (1 - threshold)

prevalence = y.mean()
print("threshold  treat-all  model-before  model-after")
for t in [0.05, 0.10, 0.20, 0.30]:
    treat_all = prevalence - (1 - prevalence) * t / (1 - t)
    print(f"{t:>8.0%}  {treat_all:>9.3f}  {net_benefit(p_before, t):>12.3f}  "
          f"{net_benefit(p_after, t):>11.3f}")

# threshold  treat-all  model-before  model-after
#       5%      0.043         0.053        0.058
#      10%     -0.010         0.028        0.040
#      20%     -0.136         0.002        0.020
#      30%     -0.299        -0.009        0.012

## Exercise

**Goal:** Detect miscalibration at a new site, recalibrate, and judge clinical usefulness with a decision curve.

**Data:** Synthetic sites generated in the notebook (no real patient data) (Generated in the notebook)

### Step 1

Reproduce the new-site calibration table and draw a smoothed calibration curve.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import SplineTransformer

print((table * 100).round(1).rename_axis("risk fifth").to_string())

def smoothed_calibration(p, outcome):
    """A flexible calibration curve: logistic regression of the outcome on a spline of the log-odds."""
    lo = np.log(p / (1 - p)).reshape(-1, 1)
    smoother = make_pipeline(SplineTransformer(n_knots=4, degree=3), LogisticRegression(penalty=None, max_iter=1000))
    smoother.fit(lo, outcome)
    grid = np.linspace(np.quantile(lo, 0.01), np.quantile(lo, 0.99), 100).reshape(-1, 1)
    return 1 / (1 + np.exp(-grid.ravel())), smoother.predict_proba(grid)[:, 1]

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(*smoothed_calibration(p_new, new_site.diabetes_5y), label="New site, smoothed")
ax.scatter(table.predicted, table.observed, label="New site, fifths of risk")
ax.plot([0, 0.7], [0, 0.7], ":", color="grey", label="Perfect calibration")
ax.set(xlabel="Predicted risk", ylabel="Observed risk")
ax.legend()
plt.show()

### Step 2

Recalibrate with only the intercept (slope fixed at 1) and with intercept and slope; compare.

In [ ]:
from scipy.optimize import brentq

def recalibrate(part, slope_too):
    lo = log_odds(part).ravel()
    if slope_too:
        r = LogisticRegression(penalty=None).fit(lo.reshape(-1, 1), part.diabetes_5y)
        return lambda lo_new: r.intercept_[0] + r.coef_[0][0] * lo_new, (r.intercept_[0], r.coef_[0][0])
    a = brentq(lambda a: np.sum(part.diabetes_5y - 1 / (1 + np.exp(-(lo + a)))), -10, 10)
    return lambda lo_new: lo_new + a, (a, 1.0)

lo_eval = log_odds(eval_part).ravel()
for label, slope_too in [("Intercept only (slope fixed at 1)", False), ("Intercept and slope", True)]:
    f, (a, b) = recalibrate(recal_part, slope_too)
    p = 1 / (1 + np.exp(-f(lo_eval)))
    print(f"{label:<34} intercept {a:+.2f}, slope {b:.2f}: mean predicted {p.mean():.1%} "
          f"(observed {eval_part.diabetes_5y.mean():.1%})")

The new site differs only in baseline risk, so the intercept does nearly all the work and the slope stays
close to 1. Fixing the slope uses one parameter instead of two, which is more stable when the local
recalibration sample is small.

### Step 3

Vary the size of the recalibration sample (200, 500, 2,000) and see how stable the result is.

In [ ]:
rows = []
pool = simulate(20000, baseline=-18.5, seed=60)      # a larger sample from the same new site
for size in [200, 500, 2000]:
    for rep in range(50):
        sample = pool.sample(size, random_state=rep)
        f, (a, _) = recalibrate(sample, slope_too=False)
        rows.append({"sample size": size, "intercept": a,
                     "mean predicted": (1 / (1 + np.exp(-f(lo_eval)))).mean()})
stability = pd.DataFrame(rows).groupby("sample size").agg(
    intercept_mean=("intercept", "mean"), intercept_sd=("intercept", "std"),
    predicted_low=("mean predicted", lambda s: s.quantile(0.05)),
    predicted_high=("mean predicted", lambda s: s.quantile(0.95)))
print(stability.round(3).to_string())

With 200 local patients (about 20 events) the recalibrated average risk can land anywhere in a wide range;
with 2,000 it is much more stable. Recalibration needs enough local outcomes, not just enough patients.

### Step 4

Plot full decision curves from 1% to 40%, with 'treat all' and 'treat none', before and after recalibration.

In [ ]:
y_eval = eval_part.diabetes_5y.to_numpy()
y = y_eval                      # net_benefit() from the worked example reads the global y
thresholds = np.arange(0.01, 0.41, 0.01)
prev = y_eval.mean()
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(thresholds, [net_benefit(p_before, t) for t in thresholds], label="Model, before recalibration")
ax.plot(thresholds, [net_benefit(p_after, t) for t in thresholds], label="Model, after recalibration")
ax.plot(thresholds, prev - (1 - prev) * thresholds / (1 - thresholds), label="Treat all")
ax.axhline(0, color="black", linewidth=1, label="Treat none")
ax.set(xlabel="Threshold probability", ylabel="Net benefit", ylim=(-0.02, prev + 0.01))
ax.legend()
plt.show()

Before recalibration the model over-predicts, so at any threshold it flags too many people and its net
benefit falls below that of the recalibrated model, dropping below zero at higher thresholds (worse than
treating nobody). The ranking is the same; only the meaning of each threshold has changed.

### Step 5

Simulate a third site where one predictor's effect is weaker, and check whether recalibration alone is enough.

In [ ]:
def simulate_weaker_glucose(n, seed):
    d = simulate(n, baseline=-18.5, seed=seed)
    rng = np.random.default_rng(seed + 100)
    logit = (-12.5 + 0.03 * d.age + 0.10 * d.bmi + 0.05 * d.fasting_glucose     # glucose effect 0.05, not 0.11
             + 0.6 * d.family_history + 0.4 * d.hypertension)
    return d.assign(diabetes_5y=rng.binomial(1, 1 / (1 + np.exp(-logit))))

site3 = simulate_weaker_glucose(5000, seed=7)
recal3, eval3 = site3.iloc[:2000], site3.iloc[2000:]
lo3 = log_odds(eval3).ravel()
p_orig = 1 / (1 + np.exp(-lo3))
f3, (a3, b3) = recalibrate(recal3, slope_too=True)
p_recal3 = 1 / (1 + np.exp(-f3(lo3)))
refit = LogisticRegression(penalty=None, max_iter=1000).fit(recal3[predictors], recal3.diabetes_5y)
p_refit = refit.predict_proba(eval3[predictors])[:, 1]

for label, p in [("Original model", p_orig), (f"Recalibrated (slope {b3:.2f})", p_recal3), ("Refitted locally", p_refit)]:
    print(f"{label:<28} AUROC {roc_auc_score(eval3.diabetes_5y, p):.3f}   mean predicted {p.mean():.1%} "
          f"(observed {eval3.diabetes_5y.mean():.1%})")

When one predictor matters less at the new site, the model ranks patients in the wrong order for that
site. Recalibration rescales all the log-odds together (a slope below 1 here), which fixes the average
risk but cannot reorder patients, so AUROC stays where it was. Refitting the coefficients (or updating the
glucose coefficient alone) recovers the lost discrimination. Recalibration fixes *calibration* problems,
not changes in how the predictors relate to the outcome.

---

Lesson: [https://ramkumarmd.com/medical-ai/machine-learning/calibration-and-decision-curves](https://ramkumarmd.com/medical-ai/machine-learning/calibration-and-decision-curves). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.